# Does the label audit actually help? Clean vs unaudited training

Two arms, identical in every respect except how unlabelled days are handled:

| Arm | Training fires | Unlabelled days |
|---|---|---|
| `clean` | 120 (audit applied) | window skipped |
| `dirty` | 139 (no audit) | window kept, label all-zero (`nan_to_num(band7) >= 7`) |

This is the experiment the earlier ablation failed to run. That version only
swapped the fire list, but `_build_index` skips windows whose last day has no
label regardless of which fires are listed, so the two arms received almost the
same data. The mechanism that actually injects false negatives is converting NaN
to 0 rather than skipping, which is what the `dirty` arm reproduces here.

Both arms are evaluated on the **test set** as well as validation, because the
claim in the manuscript is about test F1 (0.818 -> 0.855).

Validation stays clean in both arms, so the comparison isolates the training
labels alone.

Attach: the TS-SatFire dataset only. No checkpoints needed.
Runtime: about 5-6 h with both arms in parallel on 2 T4s. Fits one session.


In [ ]:
# --- Cell 1: Imports ---
import os, gc, sys, time, glob, random, warnings, json, math
from datetime import datetime
from collections import OrderedDict

PIPELINE_START = time.time()

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score
from tqdm.auto import tqdm

try:
    import rasterio
except ImportError:
    os.system("pip install rasterio --quiet")
    import rasterio

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"Device:  {DEVICE}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} -- {p.total_memory/1e9:.1f} GB")
print(f"Setup: {time.time()-PIPELINE_START:.1f}s")




In [ ]:
class Config:
    DATA_ROOT = ""
    for _p in ["/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire",
               "/kaggle/input/ts-satfire/ts-satfire/ts-satfire",
               "/kaggle/input/ts-satfire/ts-satfire",
               "/kaggle/input/ts-satfire"]:
        if os.path.isdir(_p):
            DATA_ROOT = _p
            break
    OUTPUT_DIR = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/checkpoints"

    TS_LENGTH = 2
    TRAIN_INTERVAL = 1
    IMAGE_SIZE = 256
    N_CHANNELS = 8
    MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                     294.31738, 14.625097, 276.4207, 275.16766], dtype=np.float32)
    STD = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                    24.703484, 9.878246, 40.64329, 40.7657], dtype=np.float32)

    SEED = 42
    MAX_EPOCHS = 40          # fixed budget, identical for both arms
    BATCH_SIZE = 8
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    USE_AMP = True

    FOCAL_ALPHA = 0.75
    FOCAL_GAMMA = 2.0
    DICE_WEIGHT = 0.5
    FOCAL_WEIGHT = 0.5
    DS_WEIGHT = 0.3

    ENCODER_CHANNELS = [64, 128, 256, 512]
    DROPOUT = 0.1
    SE_REDUCTION = 8

    MIN_FIRE_PX = 10
    MAX_NEG_RATIO = 2

    VAL_IDS = ["20568194", "20701026", "20562846", "20700973", "24462610",
               "24462788", "24462753", "24103571", "21998313", "21751303",
               "22141596", "21999381", "22712904"]

    NO_LABEL_IDS = [
        "20777207", "20777386", "21693566", "21751309",
        "21889672", "21889683", "21889697", "21889719",
        "21889734", "21889754", "21997775", "22712973",
        "22713339", "23860939", "23860978", "23861018",
        "23861131", "24332700", "22712904",
    ]

cfg = Config()
assert cfg.DATA_ROOT, "TS-SatFire dataset not found"
os.makedirs(cfg.SAVE_DIR, exist_ok=True)

random.seed(cfg.SEED); np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED); torch.cuda.manual_seed_all(cfg.SEED)

N_GPU = torch.cuda.device_count()
ARMS = ["clean", "dirty"]
RESULTS_JSON = os.path.join(cfg.OUTPUT_DIR, "audit_value_results.json")

print(f"DATA_ROOT: {cfg.DATA_ROOT}")
print(f"Seed {cfg.SEED} | {cfg.MAX_EPOCHS} epochs per arm | GPUs {N_GPU}")


In [ ]:
def load_frame(fire_dir, day_path, return_label=False):
    """Load 8-channel frame + optional AF label."""
    with rasterio.open(day_path) as src:
        day_arr = src.read().astype(np.float32)
    day_bands = day_arr[:6]
    label = None
    if return_label and day_arr.shape[0] >= 7:
        b7 = day_arr[6]
        if np.isnan(b7).sum() < b7.size:  # not all NaN
            label = (b7 >= 7).astype(np.float32)

    night_dir = os.path.join(fire_dir, "VIIRS_Night")
    night_path = os.path.join(night_dir,
        os.path.basename(day_path).replace("_VIIRS_Day", "_VIIRS_Night"))
    if os.path.exists(night_path):
        with rasterio.open(night_path) as src:
            na = src.read().astype(np.float32)
        nb = na[:2] if na.shape[0] >= 2 else np.zeros((2, *day_bands.shape[1:]), dtype=np.float32)
    else:
        nb = np.zeros((2, *day_bands.shape[1:]), dtype=np.float32)
    frame = np.concatenate([day_bands, nb], axis=0)
    return (frame, label) if return_label else frame


def check_day_has_label(day_path):
    """Quick check if band 7 has any non-NaN values."""
    with rasterio.open(day_path) as src:
        if src.count < 7:
            return False
        b7 = src.read(7).astype(np.float32)
        return np.isnan(b7).sum() < b7.size


# Build clean train/val splits
all_ids = sorted(os.listdir(cfg.DATA_ROOT))
numeric_ids = [d for d in all_ids if d.isdigit()]

# Exclude fires with no labels
clean_train_ids = [d for d in numeric_ids
                   if d not in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]
clean_val_ids = [d for d in numeric_ids
                 if d in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]

train_fires = [os.path.join(cfg.DATA_ROOT, d) for d in clean_train_ids]
val_fires = [os.path.join(cfg.DATA_ROOT, d) for d in clean_val_ids]

print(f"Original:  {len(numeric_ids)} fires")
print(f"Excluded:  {len(cfg.NO_LABEL_IDS)} fires (zero labels)")
print(f"Clean train: {len(train_fires)} fires")
print(f"Clean val:   {len(val_fires)} fires")
print(f"Removed from train: {len(numeric_ids) - len(cfg.VAL_IDS) - len(train_fires)} fires")
print(f"Removed from val:   {len(cfg.VAL_IDS) - len(val_fires)} fires")


class AFDatasetClean(Dataset):
    """
    Like v1's dataset but with day-level label checking.
    Skips windows where the last day has no valid label (all-NaN band 7).
    This ensures every training sample has a verified ground truth.
    """
    def __init__(self, fire_dirs, time_steps, interval, patch_size,
                 means, stds, augment=False, min_fire_px=10, max_neg_ratio=2):
        self.T = time_steps
        self.ps = patch_size
        self.means = means
        self.stds = stds
        self.augment = augment
        self.samples = []
        self._build_index(fire_dirs, interval, min_fire_px, max_neg_ratio)

    def _build_index(self, fire_dirs, interval, min_fire_px, max_neg_ratio):
        n_pos = n_neg = n_neg_kept = skipped = no_label_days = 0
        rng = random.Random(cfg.SEED)

        for i, fd in enumerate(fire_dirs):
            day_files = sorted(glob.glob(os.path.join(fd, "VIIRS_Day", "*.tif")))
            if len(day_files) < self.T:
                skipped += 1; continue
            try:
                with rasterio.open(day_files[0]) as src:
                    if src.count < 7: skipped += 1; continue
                    H, W = src.height, src.width
            except Exception:
                skipped += 1; continue
            if H < self.ps or W < self.ps:
                skipped += 1; continue

            start = 0
            while start + self.T <= len(day_files):
                last_day = day_files[start + self.T - 1]

                # KEY FIX: check if last day has valid label
                if not check_day_has_label(last_day):
                    no_label_days += 1
                    start += interval
                    continue

                lbl = None
                try:
                    with rasterio.open(last_day) as src:
                        if src.count >= 7:
                            b7 = src.read(7).astype(np.float32)
                            if np.isnan(b7).sum() < b7.size:
                                lbl = (b7 >= 7).astype(np.float32)
                except Exception: pass

                if lbl is None:
                    no_label_days += 1
                    start += interval
                    continue

                r0 = (H - self.ps) // 2; c0 = (W - self.ps) // 2
                fire_px = int(lbl[r0:r0+self.ps, c0:c0+self.ps].sum())
                is_pos = fire_px >= min_fire_px

                if is_pos:
                    n_pos += 1; keep = True
                else:
                    n_neg += 1
                    keep = rng.random() < 1.0 / (max_neg_ratio + 1)
                    if keep: n_neg_kept += 1

                if keep:
                    self.samples.append({"fd": fd, "files": day_files,
                                         "start": start, "H": H, "W": W})
                start += interval

            if (i+1) % 20 == 0 or (i+1) == len(fire_dirs):
                print(f"\r  Index: {i+1}/{len(fire_dirs)} | {len(self.samples)} samp",
                      end="", flush=True)

        print(f"\n  Done: {len(self.samples)} samples "
              f"(pos={n_pos}, neg_kept={n_neg_kept}/{n_neg}, "
              f"skip={skipped}, days_no_label={no_label_days})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fd, H, W = s["fd"], s["H"], s["W"]
        win = s["files"][s["start"]:s["start"] + self.T]

        frames, label = [], None
        for t, dp in enumerate(win):
            is_last = (t == len(win) - 1)
            if is_last:
                fr, label = load_frame(fd, dp, return_label=True)
            else:
                fr = load_frame(fd, dp)
            frames.append(fr[:, :H, :W])

        if label is None:
            label = np.zeros((H, W), dtype=np.float32)
        label = label[:H, :W]

        stack = np.stack(frames, axis=0)
        stack = (stack - self.means[None, :, None, None]) / \
                (self.stds[None, :, None, None] + 1e-8)
        stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)

        # Center crop
        r0 = (H - self.ps) // 2; c0 = (W - self.ps) // 2
        stack = stack[:, :, r0:r0+self.ps, c0:c0+self.ps]
        label = label[r0:r0+self.ps, c0:c0+self.ps]

        # Augmentation
        if self.augment:
            if random.random() > 0.5:
                stack = np.flip(stack, axis=-1).copy()
                label = np.flip(label, axis=-1).copy()
            if random.random() > 0.5:
                stack = np.flip(stack, axis=-2).copy()
                label = np.flip(label, axis=-2).copy()
            k = random.randint(0, 3)
            if k:
                stack = np.rot90(stack, k, axes=(-2, -1)).copy()
                label = np.rot90(label, k, axes=(0, 1)).copy()

        x = torch.from_numpy(stack.transpose(1, 0, 2, 3).copy()).float()
        y = torch.from_numpy(label.copy()).long()
        return x, y


print("\nBuilding CLEAN train index...")
train_ds = AFDatasetClean(train_fires, cfg.TS_LENGTH, cfg.TRAIN_INTERVAL, cfg.IMAGE_SIZE,
                          cfg.MEAN, cfg.STD, augment=True,
                          min_fire_px=cfg.MIN_FIRE_PX, max_neg_ratio=cfg.MAX_NEG_RATIO)

print("\nBuilding CLEAN val index...")
val_ds = AFDatasetClean(val_fires, cfg.TS_LENGTH, cfg.TRAIN_INTERVAL, cfg.IMAGE_SIZE,
                        cfg.MEAN, cfg.STD, augment=False,
                        min_fire_px=cfg.MIN_FIRE_PX, max_neg_ratio=cfg.MAX_NEG_RATIO)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True,
                          drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True,
                        persistent_workers=True)

print(f"\nClean train: {len(train_ds)} samples, {len(train_loader)} bat/ep")
print(f"Clean val:   {len(val_ds)} samples, {len(val_loader)} bat/ep")

xb, yb = next(iter(train_loader))
print(f"x: {tuple(xb.shape)} | y: {tuple(yb.shape)} | "
      f"y unique: {yb.unique().tolist()} | fire%: {(yb==1).float().mean():.4f}")
print(f"\nCell 3 done in {time.time()-PIPELINE_START:.0f}s")




In [ ]:
class AFDatasetArm(Dataset):
    """AF windows. The only difference between arms is unlabelled-day handling.

    clean : a window whose last day has no band-7 label is skipped
    dirty : the window is kept and the label becomes all-zero, reproducing
            nan_to_num(band7) >= 7
    """

    def __init__(self, fire_dirs, cfg, dirty, augment=False, quiet=False):
        self.T = cfg.TS_LENGTH
        self.ps = cfg.IMAGE_SIZE
        self.means, self.stds = cfg.MEAN, cfg.STD
        self.dirty = dirty
        self.augment = augment
        self.samples = []
        self._build(fire_dirs, cfg, quiet)

    def _build(self, fire_dirs, cfg, quiet):
        rng = random.Random(cfg.SEED)
        n_unlabelled = 0
        for fd in fire_dirs:
            day_files = sorted(glob.glob(os.path.join(fd, "VIIRS_Day", "*.tif")))
            if len(day_files) < self.T:
                continue
            try:
                with rasterio.open(day_files[0]) as src:
                    if src.count < 7:
                        continue
                    H, W = src.height, src.width
            except Exception:
                continue
            if H < self.ps or W < self.ps:
                continue

            start = 0
            while start + self.T <= len(day_files):
                last_day = day_files[start + self.T - 1]
                lbl, has_label = None, False
                try:
                    with rasterio.open(last_day) as src:
                        if src.count >= 7:
                            b7 = src.read(7).astype(np.float32)
                            has_label = np.isnan(b7).sum() < b7.size
                            if has_label:
                                lbl = (b7 >= 7).astype(np.float32)
                            elif self.dirty:
                                # v1 behaviour: NaN -> 0, so every pixel negative
                                lbl = np.zeros((H, W), dtype=np.float32)
                except Exception:
                    pass

                if lbl is None:
                    start += cfg.TRAIN_INTERVAL
                    continue
                if not has_label:
                    n_unlabelled += 1

                r0, c0 = (H - self.ps) // 2, (W - self.ps) // 2
                fire_px = int(lbl[r0:r0 + self.ps, c0:c0 + self.ps].sum())
                keep = (fire_px >= cfg.MIN_FIRE_PX) or (rng.random() < 1.0 / (cfg.MAX_NEG_RATIO + 1))
                if keep:
                    self.samples.append({"fd": fd, "files": day_files,
                                         "start": start, "H": H, "W": W,
                                         "labelled": has_label})
                start += cfg.TRAIN_INTERVAL

        if not quiet:
            tag = "dirty" if self.dirty else "clean"
            print(f"  [{tag}] {len(self.samples)} windows "
                  f"({n_unlabelled} from unlabelled days)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fd, H, W = s["fd"], s["H"], s["W"]
        win = s["files"][s["start"]:s["start"] + self.T]

        frames, label = [], None
        for t, dp in enumerate(win):
            if t == len(win) - 1:
                fr, label = load_frame(fd, dp, return_label=True)
            else:
                fr = load_frame(fd, dp)
            frames.append(fr[:, :H, :W])
        if label is None:
            label = np.zeros((H, W), np.float32)      # dirty arm: unlabelled day
        label = label[:H, :W]

        stack = np.stack(frames, axis=0)
        stack = (stack - self.means[None, :, None, None]) / (self.stds[None, :, None, None] + 1e-8)
        stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)

        r0, c0 = (H - self.ps) // 2, (W - self.ps) // 2
        stack = stack[:, :, r0:r0 + self.ps, c0:c0 + self.ps]
        label = label[r0:r0 + self.ps, c0:c0 + self.ps]

        if self.augment:
            if random.random() > 0.5:
                stack = np.flip(stack, -1).copy(); label = np.flip(label, -1).copy()
            if random.random() > 0.5:
                stack = np.flip(stack, -2).copy(); label = np.flip(label, -2).copy()
            k = random.randint(0, 3)
            if k:
                stack = np.rot90(stack, k, axes=(-2, -1)).copy()
                label = np.rot90(label, k, axes=(0, 1)).copy()

        x = torch.from_numpy(stack.transpose(1, 0, 2, 3).copy()).float()
        y = torch.from_numpy(label.copy()).long()
        return x, y


all_ids = sorted(os.listdir(cfg.DATA_ROOT))
numeric = [d for d in all_ids if d.isdigit()]
clean_ids = [d for d in numeric if d not in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]
dirty_ids = [d for d in numeric if d not in cfg.VAL_IDS]
val_ids   = [d for d in numeric if d in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]

P = lambda ids: [os.path.join(cfg.DATA_ROOT, d) for d in ids]
print(f"clean train fires: {len(clean_ids)} | dirty train fires: {len(dirty_ids)} "
      f"| val fires: {len(val_ids)}")

print("Building indices...")
DS = {"clean": AFDatasetArm(P(clean_ids), cfg, dirty=False, augment=True),
      "dirty": AFDatasetArm(P(dirty_ids), cfg, dirty=True,  augment=True)}
VAL_DS = AFDatasetArm(P(val_ids), cfg, dirty=False, augment=False)
print(f"  [val]   {len(VAL_DS.samples)} windows (clean in both arms)")
extra = len(DS["dirty"].samples) - len(DS["clean"].samples)
print(f"\nDifference: dirty has {extra:+d} windows ({100*extra/max(len(DS['clean'].samples),1):+.1f}%)")
if extra <= 0:
    print("WARNING: the dirty arm gained no windows. Check NO_LABEL_IDS against the data.")


In [ ]:
class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(nn.Linear(ch, ch//r, bias=False), nn.ReLU(True),
                                nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)


class ResBlock3D(nn.Module):
    def __init__(self, ic, oc, r=8, dr=0.1):
        super().__init__()
        self.c1 = nn.Conv3d(ic, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.b1 = nn.BatchNorm3d(oc)
        self.c2 = nn.Conv3d(oc, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.b2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc, r)
        self.relu = nn.ReLU(True)
        self.drop = nn.Dropout3d(dr) if dr > 0 else nn.Identity()
        self.skip = (nn.Sequential(nn.Conv3d(ic, oc, 1, bias=False),
                     nn.BatchNorm3d(oc)) if ic != oc else nn.Identity())

    def forward(self, x):
        r = self.skip(x)
        o = self.relu(self.b1(self.c1(x)))
        o = self.drop(o)
        o = self.b2(self.c2(o))
        o = self.se(o)
        return self.relu(o + r)


class SEUNet3D(nn.Module):
    def __init__(self, ic=8, nc=1, ec=(64,128,256,512), r=8, dr=0.1):
        super().__init__()
        self.e1 = ResBlock3D(ic, ec[0], r, dr)
        self.e2 = ResBlock3D(ec[0], ec[1], r, dr)
        self.e3 = ResBlock3D(ec[1], ec[2], r, dr)
        self.e4 = ResBlock3D(ec[2], ec[3], r, dr)
        self.pool = nn.MaxPool3d((1,2,2), stride=(1,2,2))
        self.bot = ResBlock3D(ec[3], ec[3]*2, r, dr)

        self.u4 = nn.ConvTranspose3d(ec[3]*2, ec[3], (1,2,2), stride=(1,2,2))
        self.d4 = ResBlock3D(ec[3]*2, ec[3], r, dr)
        self.u3 = nn.ConvTranspose3d(ec[3], ec[2], (1,2,2), stride=(1,2,2))
        self.d3 = ResBlock3D(ec[2]*2, ec[2], r, dr)
        self.u2 = nn.ConvTranspose3d(ec[2], ec[1], (1,2,2), stride=(1,2,2))
        self.d2 = ResBlock3D(ec[1]*2, ec[1], r, dr)
        self.u1 = nn.ConvTranspose3d(ec[1], ec[0], (1,2,2), stride=(1,2,2))
        self.d1 = ResBlock3D(ec[0]*2, ec[0], r, dr)

        self.final = nn.Conv3d(ec[0], nc, 1)
        self.ds3 = nn.Conv3d(ec[2], nc, 1)  # deep supervision

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.bot(self.pool(e4))

        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))

        out = self.final(d1)
        if self.training:
            ds = F.interpolate(self.ds3(d3), size=out.shape[2:],
                               mode="trilinear", align_corners=False)
            return out, ds
        return out


class DiceFocalLoss(nn.Module):
    def __init__(self, dw=0.5, fw=0.5, gamma=2.0, alpha=0.75, dsw=0.3):
        super().__init__()
        self.dw, self.fw, self.gamma, self.alpha, self.dsw = dw, fw, gamma, alpha, dsw

    def _dice(self, p, t):
        ps = torch.sigmoid(p).reshape(-1); tf = t.reshape(-1)
        return 1 - (2*(ps*tf).sum()+1) / (ps.sum()+tf.sum()+1)

    def _focal(self, p, t):
        bce = F.binary_cross_entropy_with_logits(p, t, reduction="none")
        pt = torch.sigmoid(p)*t + (1-torch.sigmoid(p))*(1-t)
        at = self.alpha*t + (1-self.alpha)*(1-t)
        return (at * (1-pt)**self.gamma * bce).mean()

    def _loss(self, p, t):
        return self.dw*self._dice(p, t) + self.fw*self._focal(p, t)

    def forward(self, preds, target):
        main = preds[0] if isinstance(preds, tuple) else preds
        ds = preds[1] if isinstance(preds, tuple) else None
        pred_last = main[:, :, -1, :, :]
        tgt = target.unsqueeze(1).float()
        loss = self._loss(pred_last, tgt)
        if ds is not None:
            loss += self.dsw * self._loss(ds[:, :, -1, :, :], tgt)
        return loss


model = SEUNet3D(ic=cfg.N_CHANNELS, nc=1, ec=tuple(cfg.ENCODER_CHANNELS),
                 r=cfg.SE_REDUCTION, dr=cfg.DROPOUT).to(DEVICE)
criterion = DiceFocalLoss(cfg.DICE_WEIGHT, cfg.FOCAL_WEIGHT,
                          cfg.FOCAL_GAMMA, cfg.FOCAL_ALPHA, cfg.DS_WEIGHT)
n_params = sum(p.numel() for p in model.parameters())

print(f"Model: SE-UNet3D v6 (identical backbone to v1)")
print(f"Params: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"Difference from v1: CLEAN DATA ONLY")
print(f"\nCell 4 done in {time.time()-PIPELINE_START:.0f}s")




In [ ]:
def run_arm(arm, device):
    torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED); random.seed(cfg.SEED)
    torch.cuda.manual_seed_all(cfg.SEED)

    model = SEUNet3D(ic=cfg.N_CHANNELS,
                     ec=tuple(cfg.ENCODER_CHANNELS),
                     r=cfg.SE_REDUCTION,
                     dr=cfg.DROPOUT).to(device)

    tl = DataLoader(DS[arm], batch_size=cfg.BATCH_SIZE, shuffle=True,
                    num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    vl = DataLoader(VAL_DS, batch_size=cfg.BATCH_SIZE, shuffle=False,
                    num_workers=cfg.NUM_WORKERS, pin_memory=True)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE,
                            weight_decay=cfg.WEIGHT_DECAY)
    sched = OneCycleLR(opt, max_lr=cfg.LEARNING_RATE, steps_per_epoch=len(tl),
                       epochs=cfg.MAX_EPOCHS, pct_start=0.1, anneal_strategy="cos")
    scaler = GradScaler(enabled=cfg.USE_AMP)
    crit = DiceFocalLoss(cfg.DICE_WEIGHT, cfg.FOCAL_WEIGHT,
                         cfg.FOCAL_GAMMA, cfg.FOCAL_ALPHA, cfg.DS_WEIGHT)

    ck = os.path.join(cfg.SAVE_DIR, f"audit_{arm}.pt")
    best_f1 = best_iou = 0.0
    best_ep = 0
    hist = []
    t0 = time.time()

    for ep in range(cfg.MAX_EPOCHS):
        model.train()
        tot = 0.0
        for xb, yb in tl:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True).float()
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=cfg.USE_AMP):
                out = model(xb)
                main, ds = (out if isinstance(out, tuple) else (out, None))
                loss = crit(main[:, 0, -1], ds[:, 0, -1] if ds is not None else None, yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()

        model.eval()
        tp = fp = fn = 0
        with torch.no_grad():
            for xb, yb in vl:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True).float()
                with autocast(enabled=cfg.USE_AMP):
                    out = model(xb)
                logits = out[0] if isinstance(out, tuple) else out
                p = torch.sigmoid(logits[:, 0, -1].float()) > 0.25
                t = yb > 0.5
                tp += int((p & t).sum()); fp += int((p & ~t).sum()); fn += int((~p & t).sum())
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)
        iou = tp / max(tp + fp + fn, 1)
        hist.append({"epoch": ep, "train_loss": tot / max(len(tl), 1),
                     "val_f1": f1, "val_iou": iou})
        if f1 > best_f1:
            best_f1, best_iou, best_ep = f1, iou, ep
            torch.save({"model_state_dict": model.state_dict(), "epoch": ep, "f1": f1}, ck)
        if (ep + 1) % 5 == 0 or ep == cfg.MAX_EPOCHS - 1:
            print(f"[{arm}] ep {ep+1}/{cfg.MAX_EPOCHS}  val F1 {f1:.4f}  best {best_f1:.4f}", flush=True)

    return {"arm": arm, "n_windows": len(DS[arm].samples),
            "best_val_f1": best_f1, "best_val_iou": best_iou,
            "best_epoch": best_ep, "hours": (time.time() - t0) / 3600.0,
            "ckpt": ck, "history": hist}


import threading
_lock = threading.Lock()
RESULTS = {}
if os.path.exists(RESULTS_JSON):
    RESULTS = json.load(open(RESULTS_JSON))
    print("already done:", list(RESULTS))

pending = [a for a in ARMS if a not in RESULTS]
print("pending arms:", pending)


def _worker(rank):
    device = torch.device("cuda", rank if N_GPU > 0 else 0)
    for a in pending[rank::max(N_GPU, 1)]:
        print(f"[gpu {rank}] starting {a}", flush=True)
        r = run_arm(a, device)
        with _lock:
            RESULTS[a] = r
            json.dump(RESULTS, open(RESULTS_JSON, "w"), indent=1)
        print(f"[gpu {rank}] finished {a}: val F1 {r['best_val_f1']:.4f}", flush=True)


if pending:
    from concurrent.futures import ThreadPoolExecutor
    n = min(max(N_GPU, 1), len(pending))
    if n > 1:
        with ThreadPoolExecutor(max_workers=n) as ex:
            for f in [ex.submit(_worker, r) for r in range(n)]:
                f.result()
    else:
        _worker(0)
print("\nTraining complete.")



In [ ]:
# Inference utilities. load_frame is already defined above.

def prepare_window(fire_dir, day_files, t_start, ts_length, mean, std, patch_size):
    """Load a T-length window, normalize, center crop. Returns (1,C,T,H,W) tensor + label."""
    frames, label = [], None
    H = W = None
    for t in range(t_start, t_start + ts_length):
        is_last = (t == t_start + ts_length - 1)
        if is_last:
            fr, label = load_frame(fire_dir, day_files[t], return_label=True)
        else:
            fr = load_frame(fire_dir, day_files[t])
        if H is None:
            H, W = fr.shape[1], fr.shape[2]
        frames.append(fr[:, :H, :W])

    if label is not None:
        label = label[:H, :W]

    stack = np.stack(frames, axis=0)  # (T, 8, H, W)
    stack = (stack - mean[None, :, None, None]) / (std[None, :, None, None] + 1e-8)
    stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)

    # Center crop
    r0 = (H - patch_size) // 2; c0 = (W - patch_size) // 2
    stack = stack[:, :, r0:r0+patch_size, c0:c0+patch_size]
    if label is not None:
        label = label[r0:r0+patch_size, c0:c0+patch_size]

    x = torch.from_numpy(stack.transpose(1, 0, 2, 3)).float().unsqueeze(0)  # (1,C,T,H,W)
    return x, label, frames[-1]  # also return raw last frame for visualization

print("Data utilities ready.")




In [ ]:
@torch.no_grad()
def evaluate_fire(fire_id, threshold=0.5):
    """Run inference on one fire, return metrics + predictions for vis."""
    fdir = os.path.join(DATA_ROOT, fire_id)
    day_files = sorted(glob.glob(os.path.join(fdir, "VIIRS_Day", "*.tif")))

    if len(day_files) < TS_LENGTH:
        return None

    tp_total = fp_total = fn_total = 0
    n_windows = 0
    n_skipped = 0
    all_probs = []
    all_labels = []
    vis_data = []  # for qualitative plots

    for t0 in range(len(day_files) - TS_LENGTH + 1):
        x, label, raw_frame = prepare_window(
            fdir, day_files, t0, TS_LENGTH, MEAN, STD, IMAGE_SIZE)

        if label is None:
            n_skipped += 1
            continue

        x = x.to(DEVICE)
        with autocast(enabled=True):
            logits = model(x)
        probs = torch.sigmoid(logits[:, 0, -1]).cpu().numpy()[0]  # (H, W)
        pred = (probs > threshold).astype(np.float32)
        lbl = label.astype(np.float32)

        tp = int(((pred == 1) & (lbl == 1)).sum())
        fp = int(((pred == 1) & (lbl == 0)).sum())
        fn = int(((pred == 0) & (lbl == 1)).sum())

        tp_total += tp; fp_total += fp; fn_total += fn
        n_windows += 1
        all_probs.append(probs.flatten())
        all_labels.append(lbl.flatten())

        # Save last window for visualization
        vis_data.append({
            "raw": raw_frame, "label": lbl, "pred": pred, "probs": probs,
            "date": os.path.basename(day_files[t0 + TS_LENGTH - 1]).replace("_VIIRS_Day.tif", "")
        })

    if n_windows == 0:
        return None

    prec = tp_total / max(tp_total + fp_total, 1)
    rec = tp_total / max(tp_total + fn_total, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    iou = tp_total / max(tp_total + fp_total + fn_total, 1)

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    return {
        "fire_id": fire_id, "n_windows": n_windows, "n_skipped": n_skipped,
        "tp": tp_total, "fp": fp_total, "fn": fn_total,
        "f1": f1, "iou": iou, "precision": prec, "recall": rec,
        "probs": all_probs, "labels": all_labels,
        "vis": vis_data,
    }



AF_TEST_ALL = [
    "elephant_hill_fire", "eagle_bluff_fire", "double_creek_fire", "sparks_lake_fire",
    "lytton_fire", "chuckegg_creek_fire", "swedish_fire", "sydney_fire",
    "thomas_fire", "tubbs_fire", "carr_fire", "camp_fire",
    "creek_fire", "blue_ridge_fire", "dixie_fire", "mosquito_fire", "calfcanyon_fire",
]
DATA_ROOT = cfg.DATA_ROOT
TS_LENGTH = cfg.TS_LENGTH
IMAGE_SIZE = cfg.IMAGE_SIZE
MEAN, STD = cfg.MEAN, cfg.STD
DEVICE0 = torch.device("cuda", 0) if N_GPU else torch.device("cpu")


def val_threshold(m, device):
    """Sweep the threshold on validation, exactly as in the main notebook."""
    m.eval()
    P, L = [], []
    vl = DataLoader(VAL_DS, batch_size=cfg.BATCH_SIZE, shuffle=False,
                    num_workers=cfg.NUM_WORKERS)
    with torch.no_grad():
        for xb, yb in vl:
            xb = xb.to(device)
            with autocast(enabled=cfg.USE_AMP):
                out = m(xb)
            lg = out[0] if isinstance(out, tuple) else out
            P.append(torch.sigmoid(lg[:, 0, -1].float()).cpu().numpy().ravel())
            L.append(yb.numpy().ravel())
    P = np.concatenate(P); L = np.concatenate(L)
    best_t, best_f = 0.5, -1
    for t in np.arange(0.05, 0.91, 0.02):
        p = P > t
        tp = int((p & (L > 0.5)).sum()); fpp = int((p & (L <= 0.5)).sum())
        fnn = int((~p & (L > 0.5)).sum())
        f = 2 * tp / max(2 * tp + fpp + fnn, 1)
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t, best_f


SUMMARY = {}
for arm in ARMS:
    if arm not in RESULTS:
        continue
    print(f"\n{'='*70}\nTEST EVALUATION -- {arm}\n{'='*70}")
    model = SEUNet3D(ic=cfg.N_CHANNELS,
                     ec=tuple(cfg.ENCODER_CHANNELS),
                     r=cfg.SE_REDUCTION,
                     dr=cfg.DROPOUT).to(DEVICE0)
    ck = torch.load(RESULTS[arm]["ckpt"], map_location=DEVICE0, weights_only=False)
    model.load_state_dict(ck["model_state_dict"])
    model.eval()

    thr, vf1 = val_threshold(model, DEVICE0)
    print(f"validation-selected threshold {thr:.2f} (val F1 {vf1:.4f})")

    tp = fp = fn = 0
    per_fire = []
    for fid in AF_TEST_ALL:
        r = evaluate_fire(fid, thr)
        if r is None:
            continue
        tp += r["tp"]; fp += r["fp"]; fn += r["fn"]
        per_fire.append({"arm": arm, "fire_id": r["fire_id"], "f1": r["f1"],
                         "iou": r["iou"], "tp": r["tp"], "fp": r["fp"], "fn": r["fn"]})
        print(f"  {fid:<25s} F1 {r['f1']:.4f}  IoU {r['iou']:.4f}")

    f1 = 2 * tp / max(2 * tp + fp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    SUMMARY[arm] = {"val_threshold": thr, "val_f1": RESULTS[arm]["best_val_f1"],
                    "test_micro_f1": f1, "test_micro_iou": iou,
                    "n_fires": len(per_fire), "n_windows": RESULTS[arm]["n_windows"],
                    "hours": RESULTS[arm]["hours"], "per_fire": per_fire}
    print(f"  TEST micro F1 {f1:.4f}  IoU {iou:.4f}  over {len(per_fire)} fires")



In [ ]:
print("\n" + "=" * 74)
print("THE VALUE OF THE LABEL AUDIT")
print("=" * 74)
print(f"{'arm':<8} {'train win':>10} {'val F1':>9} {'thr':>6} {'test F1':>9} {'test IoU':>9}")
print("-" * 74)
for a in ARMS:
    if a not in SUMMARY:
        continue
    s = SUMMARY[a]
    print(f"{a:<8} {s['n_windows']:>10} {s['val_f1']:>9.4f} {s['val_threshold']:>6.2f} "
          f"{s['test_micro_f1']:>9.4f} {s['test_micro_iou']:>9.4f}")

if "clean" in SUMMARY and "dirty" in SUMMARY:
    dv = SUMMARY["clean"]["val_f1"] - SUMMARY["dirty"]["val_f1"]
    dt = SUMMARY["clean"]["test_micro_f1"] - SUMMARY["dirty"]["test_micro_f1"]
    print("-" * 74)
    print(f"Audit is worth {dv:+.4f} validation F1 and {dt:+.4f} test F1")
    print(f"AF seed noise (sigma) = 0.0009; 3 sigma = 0.0028")
    verdict = "SIGNIFICANT" if abs(dt) > 0.0028 else "within noise"
    print(f"Test effect is {abs(dt)/0.0009:.1f} sigma -> {verdict}")
    print()
    if abs(dt) > 0.0028:
        print("Report as: 'excluding fires with unusable labels improves test F1 by "
              f"{dt:.4f}'.")
    else:
        print("The audit does not produce a measurable F1 gain under this loader.")
        print("Report it as preventing a failure mode rather than as an F1 improvement,")
        print("and state the measured effect honestly.")

pd.DataFrame([r for a in SUMMARY for r in SUMMARY[a]["per_fire"]]).to_csv(
    os.path.join(cfg.OUTPUT_DIR, "audit_value_per_fire.csv"), index=False)
out = {a: {k: v for k, v in s.items() if k != "per_fire"} for a, s in SUMMARY.items()}
json.dump(out, open(os.path.join(cfg.OUTPUT_DIR, "audit_value_summary.json"), "w"), indent=2)
print("\nWrote audit_value_summary.json and audit_value_per_fire.csv")
print("Send me audit_value_summary.json.")
